# Urban Expansion Analysis — Canary Islands

Detects urban growth on Gran Canaria using **Google Dynamic World** `label` band with `mode()` temporal aggregation.

**Method:** For each pixel, the most frequent land-cover class across all scenes in the period is selected (`mode`). This is more stable than probability thresholding, which is sensitive to outlier scenes. Class 6 = Built, Class 0 = Water (masked for coastal accuracy).

**Output:** Three categories — Stable Urban, New Growth, Loss/Flicker — with area statistics in km².

In [4]:
import ee
import geemap

ee.Initialize(project='sensiblesat')
# To profile a GEE call, wrap it in: with ee.profilePrinting(): ...
# e.g. with ee.profilePrinting(): print(get_km2(mask, roi).getInfo())

In [5]:
# --- Configuration ---
# ROI: Gran Canaria (30 km buffer around center point)
roi = ee.Geometry.Point([-15.5474, 27.9202]).buffer(30000)

# Early period (full year for good mode coverage)
EARLY_START = '2016-01-01'
EARLY_END   = '2016-12-31'

# Recent period
RECENT_START = '2025-01-01'
RECENT_END   = '2026-02-01'

In [6]:
def get_stable_label_layer(roi, start_date, end_date):
    """Return a binary built mask using Dynamic World label mode.

    For each pixel, mode() selects the most frequent class across all
    scenes in the date range, filtering out transient misclassifications.
    Water pixels (class 0) are masked to avoid coastal false positives.
    """
    collection = (
        ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
        .filterBounds(roi)
        .filterDate(start_date, end_date)
    )
    mode_image = collection.select('label').mode().clip(roi)

    built_mask = mode_image.eq(6)   # Class 6 = Built
    water_mask = mode_image.eq(0)   # Class 0 = Water

    return built_mask.updateMask(water_mask.Not())

In [7]:
def get_km2(mask, roi):
    """Convert a binary mask to area in km², using scale 30 m for speed."""
    stats = (
        mask.multiply(ee.Image.pixelArea())
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=roi,
            scale=30,
            maxPixels=1e10,
        )
    )
    return ee.Number(stats.get('label')).divide(1e6)

In [8]:
# --- Generate masks ---
mask_early  = get_stable_label_layer(roi, EARLY_START, EARLY_END)
mask_recent = get_stable_label_layer(roi, RECENT_START, RECENT_END)

# --- Categorize ---
stable     = mask_early.And(mask_recent)
new_growth = mask_recent.And(mask_early.Not())
removed    = mask_early.And(mask_recent.Not())

# --- Statistics ---
with ee.profilePrinting():
    area_stable = get_km2(stable, roi).getInfo()
    area_new    = get_km2(new_growth, roi).getInfo()
    area_loss   = get_km2(removed, roi).getInfo()

print(f'Stable Urban:  {area_stable:.2f} km\u00b2')
print(f'New Growth:    {area_new:.2f} km\u00b2')
print(f'Loss/Flicker:  {area_loss:.2f} km\u00b2')
print(f'Net Change:    {area_new - area_loss:+.2f} km\u00b2')

Stable Urban:  284.64 km²
New Growth:    43.26 km²
Loss/Flicker:  23.13 km²
Net Change:    +20.12 km²


 EECU·s PeakMem Count  Description
1646.468    213k   360  Algorithm reduce.mode
 53.453    382M 113856  Loading assets: GOOGLE/DYNAMICWORLD/V1_RAW/(...)
 19.010    7.0M 188988  (plumbing)
  9.605    1.0M    96  Algorithm Image.reduceRegion
  8.275    2.8M 82500  Algorithm Image.select
  6.630     16M    36  Reprojection precalculation between EPSG:32628 and EPSG:4326
  5.939     16M    36  Algorithm Image.pixelArea computing pixels
  4.905    3.7M 41250  Algorithm Image.addBands
  2.702    3.2M 20625  Algorithm Image.double
  2.680    3.7M 20625  Algorithm Image.divide
  2.617    3.3M 20625  Algorithm Image.pow
  0.930    1.5k 13698  Reprojecting pixels from EPSG:32628 to EPSG:4326
  0.096     14k    72  no description available
  0.037     14k   144  Algorithm Image.clip
  0.012     352    72  Algorithm Image.constant computing pixels
  0.011    8.3k   288  Algorithm Image.eq
  0.010    2.5k   144  Algorithm Image.clip computing pixels
  0.010    4.2k   504  Algorithm Image.constant


In [9]:
# --- Visualization ---
Map = geemap.Map()
Map.clear_layers()
Map.add_basemap('Stadia.StamenTerrain')

Map.addLayer(stable.selfMask(),     {'palette': 'FFFFFF'}, 'Stable Urban (White)')
Map.addLayer(new_growth.selfMask(), {'palette': '00FF00'}, 'New Growth (Green)')
Map.addLayer(removed.selfMask(),    {'palette': 'FF0000'}, 'Loss/Flicker (Red)')

Map.add_text('White: Stable | Green: New | Red: Loss', position='bottomleft')
Map.centerObject(roi, 11)
Map

Map(center=[27.920230776216357, -15.547399034103881], controls=(WidgetControl(options=['position', 'transparen…